In [33]:
import dolfinx as dfx
import multiphenicsx.fem
from mpi4py import MPI
from petsc4py.PETSc import ScalarType
import numpy as np
import ufl
from solver import NonlinearBlockProblem, NewtonBlockSolver
import matplotlib.pyplot as plt
from cideMOD.numerics.helper import print_diagonal_statistics, plot_jacobian

n= 40
stretch = 300
mesh = dfx.mesh.create_box(MPI.COMM_WORLD, [(0,0,0),(1,1*stretch,1*stretch)], [n, n, n])
# mesh = dfx.mesh.create_rectangle(MPI.COMM_WORLD, [(0,0),(1,1*stretch)], [n, n])
cdim = mesh.topology.dim
fdim = mesh.topology.dim - 1

# Define boundaries
boundaries = [(1, lambda x: np.isclose(x[0], 0)),
                (2, lambda x: np.isclose(x[0], 1)),]
left_facets = dfx.mesh.locate_entities(mesh, fdim, boundaries[0][1]) 
right_facets = dfx.mesh.locate_entities(mesh, fdim, boundaries[1][1])
facet_indices, facet_markers = [], []
for (marker, locator) in boundaries:
    facets = dfx.mesh.locate_entities(mesh, fdim, locator)
    facet_indices.append(facets)
    facet_markers.append(np.full(len(facets), marker))
facet_indices = np.array(np.hstack(facet_indices), dtype=np.int32)
facet_markers = np.array(np.hstack(facet_markers), dtype=np.int32)
sorted_facets = np.argsort(facet_indices)
facet_tag = dfx.mesh.MeshTags(mesh, fdim, facet_indices[sorted_facets], facet_markers[sorted_facets])

# Define domains
domains = [
    (1, lambda x: x[0]<=0.2),
    (2, lambda x: x[0]>=0.2),
]
left_cells = dfx.mesh.locate_entities(mesh, cdim, domains[0][1]) 
right_cells = dfx.mesh.locate_entities(mesh, cdim, domains[1][1])
cell_indices, cell_markers = [], []
for (marker, locator) in domains:
    cells = dfx.mesh.locate_entities(mesh, cdim, locator)
    cell_indices.append(cells)
    cell_markers.append(np.full(len(cells), marker))
cell_indices = np.array(np.hstack(cell_indices), dtype=np.int32)
cell_markers = np.array(np.hstack(cell_markers), dtype=np.int32)
sorted_cells = np.argsort(cell_indices)
cell_tag = dfx.mesh.MeshTags(mesh, cdim, cell_indices[sorted_cells], cell_markers[sorted_cells])

# Define function spaces
PHI_E = dfx.fem.FunctionSpace(mesh, ('CG',1))
PHI_S = PHI_E.clone()
LM_IAPP = PHI_E.clone()
phi_e, phi_s, lm_iapp = dfx.fem.Function(PHI_E, name='phi_e'), dfx.fem.Function(PHI_S, name='phi_s'), dfx.fem.Function(LM_IAPP,name='lm_iapp')
dphi_e, dphi_s, dlm_iapp = ufl.TrialFunction(PHI_E), ufl.TrialFunction(PHI_S), ufl.TrialFunction(LM_IAPP)
v_phi_e, v_phi_s, v_lm_iapp = ufl.TestFunction(PHI_E), ufl.TestFunction(PHI_S), ufl.TestFunction(LM_IAPP)
# Define measures
dx = ufl.Measure('dx', mesh, subdomain_data=cell_tag)
ds = ufl.Measure('ds', mesh, subdomain_data=facet_tag)
# Define constants 
I_app = dfx.fem.Constant(mesh, ScalarType(1))
V_app = dfx.fem.Constant(mesh, ScalarType(1))
switch = dfx.fem.Constant(mesh, ScalarType(0))


# Define WF
L=20e-6; ocv=1
reaction = 1e-5*2*ufl.sinh(0.5*94685/(8.3*298)*(phi_s-phi_e-ocv))
residuals = [
    ufl.inner(ufl.grad(phi_e),ufl.grad(v_phi_e)) * dx - L*lm_iapp*v_phi_e*ds(2) - L**2*reaction*v_phi_e*dx(1),
    1e2*ufl.inner(ufl.grad(phi_s),ufl.grad(v_phi_s)) * dx(1) + L**2*reaction*v_phi_s*dx(1),
    (1-switch)*(I_app-lm_iapp)*v_lm_iapp*ds(2) + switch*(V_app-phi_e)*v_lm_iapp*ds(2)
]
jacobian = [[ufl.derivative(r_i, u_i, du_i) for u_i, du_i in zip([phi_e, phi_s, lm_iapp],[dphi_e, dphi_s, dlm_iapp])] for r_i in residuals]
bc = dfx.fem.dirichletbc(dfx.fem.Constant(mesh, ScalarType(0)), dfx.fem.locate_dofs_topological(PHI_S, fdim, left_facets), PHI_S)
restrictions = [
    multiphenicsx.fem.DofMapRestriction(PHI_E.dofmap, np.arange(0, PHI_E.dofmap.index_map.size_local+PHI_E.dofmap.index_map.num_ghosts)),
    multiphenicsx.fem.DofMapRestriction(PHI_S.dofmap, dfx.fem.locate_dofs_topological(PHI_S, cdim, left_cells)), 
    multiphenicsx.fem.DofMapRestriction(LM_IAPP.dofmap, dfx.fem.locate_dofs_topological(LM_IAPP, fdim, right_facets))
]

# init phi_s
noise = np.random.random(phi_s.vector.array.size)-0.5
phi_s.vector.array=ocv

# Solve
problem = NonlinearBlockProblem(residuals, [phi_e, phi_s, lm_iapp], [bc], jacobian, restrictions)
solver = NewtonBlockSolver(mesh.comm, problem, pc='hypre')
solver.solve()


--------------- Problem Residuals ---------------
      objective        phi_e        phi_s         lm_iapp
--  -----------  -----------  -----------  --------------
 0  3.55503e+07  1.26956e-11  3.55503e+07  2221.89
 1  3.55503e+07  0.0284302    3.55503e+07  2221.89
 2  5.73294      0.0152182    5.73292         7.79226e-05
 3  5.69714      0.0152137    5.69712         7.74352e-05
 4  5.54641      0.520804     5.5219          7.74352e-05
 5  0.263092     0.187671     0.184383        1.05281e-08
 6  0.0914741    0.0652293    0.06413         1.82578e-13
 7  0.0288052    0.0205407    0.0201945       1.11546e-13
--------------------- End -----------------------

--------------- Problem Steps ---------------
      objective     phi_e          phi_s       lm_iapp
--  -----------  --------  -------------  ------------
 0  3.55503e+07    0         0             0
 1  3.55503e+07  305.176     1.23218e-05   1.01286e-06
 2  5.73294      961.907   492            41
 3  5.69714      289.036     4.9

(7, True)

In [35]:
import pyvista
pyvista.start_xvfb()
pyvista.set_jupyter_backend('panel')
cells, types, x = dfx.plot.create_vtk_mesh(PHI_E)
x[:,0]=x[:,0]*stretch
grid = pyvista.UnstructuredGrid(cells, types, x)
grid.point_data["u"] = phi_e.x.array.real*stretch
grid.set_active_scalars("u")
plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True)
# warped = grid.warp_by_scalar()
# plotter.add_mesh(warped, show_edges=True)
plotter.show()

In [31]:
print_diagonal_statistics(problem)

--------------- Diagonal values ---------------
Fields            Max      Min
--------  -----------  -------
phi_e       4500.1     375.013
phi_s     450010         1
lm_iapp       -4.6875  -28.125
--------------------- End ----------------------
